In [ ]:
sys.path.insert(0, snakemake.input.data2dd)

In [ ]:
import numpy as np
import pandas as pd
from data2dd_funcs import wrapdd

In [ ]:
year = snakemake.wildcards.year
date_range = snakemake.params.date_range
date_range = [year + "-" + date for date in date_range]
date_range = pd.date_range(date_range[0], date_range[1] + ' 23', freq='h')

In [ ]:
aggregated_regions = snakemake.params.aggregated_regions
aggregated_regions

In [ ]:
outdd = []

In [ ]:
# ev_scenarios.csv # contain the number of EVs and battery capacity per zone
ev_scenarios = snakemake.input["ev_scenarios"]

evs = pd.read_csv(ev_scenarios, index_col=(0,1)).loc[snakemake.params.ev_scenario[0], 'evs'][aggregated_regions].T.reset_index()
outdd.append(wrapdd(evs, "par_vehicles", "parameter"))

ecap = pd.read_csv(ev_scenarios, index_col=(0,1)).loc[snakemake.params.ev_scenario[0], 'ecap'][aggregated_regions].T.reset_index()
outdd.append(wrapdd(ecap, "par_ev_ecap", "parameter"))

In [ ]:
# EV_grid_connected_power_kW.csv

s_ev_pcap = pd.read_csv(snakemake.input[3], sep=' ', header=None, index_col=0).iloc[:,0].str.strip('/').astype(float).loc['s_ev_pcap']

connected_vehicles = pd.read_csv(snakemake.input[0])
connected_vehicles = connected_vehicles.charging_cap.iloc[:len(date_range)].round(3).div(1000).div(s_ev_pcap).reset_index().astype({'index':str})
outdd.append(wrapdd(connected_vehicles, "par_grid_connected", "parameter"))

In [ ]:
# demand_driving_MWh.tsv
demand_driving = pd.read_csv(snakemake.input[1])
demand_driving = demand_driving.set_index(demand_driving.reset_index()["index"].astype('str') + '.EV')['battery discharge kWh'].iloc[:len(date_range)].div(1000).reset_index()
outdd.append(wrapdd(demand_driving, "par_driving_demand", "parameter"))

In [ ]:
# demand_ev_charging_MWh.tsv
demand_ev_charging = pd.read_csv(snakemake.input[2])
demand_ev_charging = demand_ev_charging.charge_battery.iloc[:len(date_range)].round(6).div(1000).reset_index().astype({'index':str})
outdd.append(wrapdd(demand_ev_charging, "par_ev_charging", "parameter"))

In [ ]:
outdd = np.concatenate(outdd, axis=0)

In [ ]:
np.savetxt(snakemake.output[0], outdd, delimiter=" ", fmt="%s")